# Notebook 05: Alignment with DPO

Fine-tuning taught the model what to say. This notebook is about teaching it to say things well. The technical term for that is alignment.

## Why this step exists

A fine-tuned model knows the facts and the style, but it can still answer in ways people do not actually like. After fine-tuning, a model might:

- Be correct but ramble on far too long
- Hedge constantly ("it depends", "hard to say") instead of committing
- Use the wrong tone for the job
- Now and then say something misleading

Alignment is the process of nudging the model toward answers that people genuinely prefer, not just answers that are technically right.

## The classic way: RLHF

The well-known method is RLHF, Reinforcement Learning from Human Feedback. It goes like this:

1. Collect preferences. Show people two answers and ask which is better.
2. Train a "reward model" that learns to predict which answer people will prefer.
3. Use reinforcement learning to push the main model toward higher-scoring answers.

This is roughly how ChatGPT and Claude were aligned. The downside is that it is fiddly and unstable. It involves juggling several models at once and is hard to get right.

## The simpler way: DPO

DPO, Direct Preference Optimisation, reaches the same goal with far less hassle.

Instead of training a separate reward model and running reinforcement learning, DPO turns the whole thing into a straightforward training task. For each prompt you give it a better answer and a worse answer, and it directly teaches the model to:

- make the better answer more likely, and
- make the worse answer less likely,

while a built-in brake stops the model from drifting too far from how it started. No reward model, no reinforcement learning, and it trains in a single pass, which is why it runs on ordinary hardware.

## What the data looks like

DPO needs preference pairs. For each prompt, one preferred answer (chosen) and one weaker answer (rejected):

```json
{
  "prompt": "What was Apple's total revenue in 2023?",
  "chosen": "Apple reported total net sales of $383.3 billion for fiscal 2023, down 2.8% from $394.3 billion in 2022, mainly on weaker iPhone demand in China.",
  "rejected": "Apple made a lot of money in 2023. Revenue was very high and the company did well overall."
}
```

In a real project these pairs come from people comparing answers, from a stronger model judging a weaker one, or from automatic quality checks.


In [ ]:
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, PeftModel
from trl import DPOTrainer, DPOConfig

MODEL_DIR = Path("../models")

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

## 1. Building a preference dataset

Each item in the dataset is a trio: a prompt, a chosen (good) answer, and a rejected (worse) answer.

The next cell builds a small, realistic financial set. The good answers are specific, backed by numbers, and the right length. The weak answers are vague, full of hedging, and light on detail. By showing the model many of these pairs, we teach it which habits to lean into and which to drop.


In [ ]:
# Preference dataset: each entry has a prompt, a good response (chosen), and a poor response (rejected)
# These represent the kinds of quality distinctions that matter in financial analysis

preference_data = [
    {
        "prompt": "What was Apple's total revenue for fiscal year 2023?",
        "chosen": "Apple reported total net sales of $383.3 billion for fiscal year 2023, a decrease of 2.8% from $394.3 billion in fiscal 2022. The decline was primarily driven by lower iPhone demand, partially offset by growth in Services revenue which rose 16% to $85.2 billion.",
        "rejected": "Apple had a lot of revenue in 2023. It was a big company and made lots of money. The exact figure might have changed so I'd recommend checking the latest reports."
    },
    {
        "prompt": "Summarise the key risks Microsoft identifies in its 10-K filing.",
        "chosen": "Microsoft's 10-K identifies five primary risk categories: (1) competitive threats from firms like Google, Amazon, and Apple in cloud and productivity; (2) cybersecurity and data privacy risks given the volume of enterprise data processed; (3) regulatory and antitrust risk across its global operations; (4) dependence on continued Azure adoption for growth; and (5) macroeconomic exposure through enterprise IT spending cycles.",
        "rejected": "Microsoft has many risks. Competition is one risk. They also have technology risks and legal risks. There are many other risks as well that are described in the document."
    },
    {
        "prompt": "How should an analyst interpret a decline in gross margin for a technology company?",
        "chosen": "A gross margin decline in a technology company warrants investigation across three dimensions: (1) product mix shift - lower-margin hardware growing faster than software/services; (2) cost pressures - component price increases, supply chain disruptions, or higher cloud infrastructure costs; and (3) competitive pricing - discounting to defend market share. The interpretation depends on whether the decline is structural (mix shift to a lower-margin business) or cyclical (temporary cost pressures expected to normalise).",
        "rejected": "A decline in gross margin could mean different things. It might be bad or it could be okay depending on the context. You should look at why it happened. Sometimes margins go down and sometimes they go up. It's hard to say without more information."
    },
    {
        "prompt": "What does JPMorgan's net interest income tell us about their business model?",
        "chosen": "JPMorgan's net interest income (NII) - the spread between what it earns on loans/investments and what it pays on deposits - is a core profitability driver for its retail and commercial banking segments. Rising NII in a high-rate environment reflects the bank's asset-sensitive balance sheet: assets (loans) reprice upward faster than liabilities (deposits). This makes JPMorgan a beneficiary of rate increases but exposes it to NII compression if rates fall or deposit outflows accelerate.",
        "rejected": "Net interest income is money that JPMorgan makes from interest. They get interest from loans and pay interest on deposits. The difference is their profit. This is important for banks."
    },
    {
        "prompt": "Explain the difference between revenue growth and earnings growth.",
        "chosen": "Revenue growth measures the rate at which a company's top-line sales are increasing. Earnings growth (typically EPS growth) measures profit growth after all costs are deducted. The gap between the two reflects margin dynamics: a company growing revenue 15% but earnings only 5% is seeing margin compression - costs rising faster than sales. Conversely, earnings growing faster than revenue signals operating leverage or improving cost efficiency. For growth-stage companies, revenue growth is often prioritised; for mature companies, earnings growth matters more to investors.",
        "rejected": "Revenue is the total sales and earnings is the profit. Revenue growth means sales are going up and earnings growth means profits are going up. Both are important financial metrics that investors look at."
    },
    {
        "prompt": "What are the trade-offs between using RAG versus fine-tuning for a financial document assistant?",
        "chosen": "RAG is preferable when: (1) the knowledge base changes frequently (quarterly filings, live prices); (2) auditability matters - you need to show which document grounded each answer; or (3) setup speed is critical. Fine-tuning is preferable when: (1) consistent analytical style and tone are important; (2) the query types are predictable and repetitive; or (3) inference latency is constrained and retrieval overhead is unacceptable. In practice, production financial assistants often combine both: fine-tuning for style and domain reasoning, RAG for live document retrieval.",
        "rejected": "RAG and fine-tuning are both ways to improve language models. RAG retrieves documents and fine-tuning trains the model. Both have pros and cons. You should choose based on your needs."
    },
]

dataset = Dataset.from_list(preference_data)
print(f"Preference dataset: {len(dataset)} examples")
print()
print("Example entry:")
print(f"  Prompt  : {dataset[0]['prompt']}")
print(f"  Chosen  : {dataset[0]['chosen'][:100]}...")
print(f"  Rejected: {dataset[0]['rejected'][:100]}...")

## 2. What makes an answer "chosen" rather than "rejected"

Before training, it is worth looking at what actually separates the good answers from the weak ones in our data. The next cell compares them, for example on length, and spells out the qualities we are rewarding: being specific, being well structured, explaining the why and not just the what, and using a professional tone.


In [ ]:
import pandas as pd

df = dataset.to_pandas()
df["chosen_len"]   = df["chosen"].str.split().str.len()
df["rejected_len"] = df["rejected"].str.split().str.len()

print("Response length comparison (words):")
print(f"  Chosen   - mean: {df['chosen_len'].mean():.0f}, min: {df['chosen_len'].min()}, max: {df['chosen_len'].max()}")
print(f"  Rejected - mean: {df['rejected_len'].mean():.0f}, min: {df['rejected_len'].min()}, max: {df['rejected_len'].max()}")
print()
print("Quality dimensions our preference data teaches:")
print("  OK Specificity     - chosen responses include concrete numbers and facts")
print("  OK Structure       - chosen responses use enumeration, categories, comparisons")
print("  OK Depth           - chosen responses explain *why*, not just *what*")
print("  OK Appropriate tone - chosen responses match professional analyst register")
print("  x Avoided: hedging, vagueness, lack of domain knowledge")

## 3. The reward model (for understanding)

DPO does not need a reward model, but it is worth building one once just to see the idea it replaced. A reward model is a separate network that reads a prompt and an answer and gives back a single number: how good the answer is.

This is still the heart of the older RLHF approach. The next few cells train a small one and check whether it gives the chosen answers higher scores than the rejected ones. Think of it as a quick detour to make the concept concrete, not a required step for our final model.


In [ ]:
from transformers import AutoModelForSequenceClassification

# A reward model is typically a language model with a classification head
# It takes (prompt + response) as input and outputs a single number: the reward
# We use a small model for demonstration

REWARD_MODEL_ID = "distilbert-base-uncased"

reward_tokeniser = AutoTokenizer.from_pretrained(REWARD_MODEL_ID)
reward_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_ID,
    num_labels=1,          # Single output: the scalar reward
)
reward_model = reward_model.to(device)

print(f"Reward model: {REWARD_MODEL_ID}")
print(f"Output: single scalar reward score")
print(f"Parameters: {sum(p.numel() for p in reward_model.parameters())/1e6:.0f}M")

In [ ]:
# Prepare training data for the reward model
# Each example: concatenate prompt + response, label = 1 (chosen) or 0 (rejected)

reward_examples = []
for ex in preference_data:
    reward_examples.append({
        "text": ex["prompt"] + " [SEP] " + ex["chosen"],
        "label": 1.0   # preferred
    })
    reward_examples.append({
        "text": ex["prompt"] + " [SEP] " + ex["rejected"],
        "label": 0.0   # not preferred
    })

reward_dataset = Dataset.from_list(reward_examples)

def tokenise_reward(examples):
    return reward_tokeniser(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenised_reward = reward_dataset.map(tokenise_reward, batched=True)
tokenised_reward = tokenised_reward.rename_column("label", "labels")
tokenised_reward.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Reward model training set: {len(tokenised_reward)} examples ({len(preference_data)} chosen + {len(preference_data)} rejected)")

In [ ]:
from transformers import Trainer, TrainingArguments

reward_training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "reward_model"),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    logging_steps=5,
    save_strategy="no",
    use_mps_device=(device == "mps"),
    fp16=False,
    bf16=False,
    report_to="none",
)

reward_trainer = Trainer(
    model=reward_model,
    args=reward_training_args,
    train_dataset=tokenised_reward,
)

print("Training reward model...")
reward_trainer.train()
print("Reward model trained.")

In [ ]:
# Test the reward model: does it assign higher scores to chosen vs rejected responses?

def get_reward(prompt, response):
    text = prompt + " [SEP] " + response
    inputs = reward_tokeniser(
        text, return_tensors="pt", truncation=True, max_length=256
    ).to(device)
    with torch.no_grad():
        output = reward_model(**inputs)
    return output.logits.squeeze().item()

print("Reward model scoring (higher = more preferred):")
print()
for ex in preference_data[:3]:
    r_chosen   = get_reward(ex["prompt"], ex["chosen"])
    r_rejected = get_reward(ex["prompt"], ex["rejected"])
    correct = "OK" if r_chosen > r_rejected else "x"
    print(f"{correct} Prompt: {ex['prompt'][:60]}...")
    print(f"    Chosen score  : {r_chosen:+.3f}")
    print(f"    Rejected score: {r_rejected:+.3f}")
    print()

## 4. The actual DPO training

Now the real thing. We take the fine-tuned model from notebook 04 and run DPO on it using our preference pairs. This directly teaches it to favour the chosen style over the rejected one, with no reward model and no reinforcement learning involved.

One setting to know about is `beta`. It is the brake mentioned earlier. A higher beta keeps the model closer to how it started, a lower beta lets it move more freely toward the preferences. We use a middle value.


In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = MODEL_DIR / "tinyllama-financial-lora" / "final_adapter"

dpo_tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)
if dpo_tokeniser.pad_token is None:
    dpo_tokeniser.pad_token = dpo_tokeniser.eos_token

# Load the fine-tuned model from Notebook 04 as our starting point
# (If notebook 04 hasn't been run, load the base model instead)
if adapter_path.exists():
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    dpo_model = PeftModel.from_pretrained(base, str(adapter_path))
    dpo_model = dpo_model.merge_and_unload()  # Merge adapter into base for DPO training
    print("Loaded fine-tuned model from Notebook 04.")
else:
    dpo_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    print("Loaded base model (run Notebook 04 first for the fine-tuned version).")

dpo_model = dpo_model.to(device)

In [ ]:
# DPO also uses LoRA adapters - we don't want to full-fine-tune the model again
dpo_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                   # Smaller rank for alignment stage
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

dpo_config = DPOConfig(
    output_dir=str(MODEL_DIR / "tinyllama-dpo"),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,              # KL penalty - higher = model stays closer to original. Typical range: 0.01-0.5
    max_length=512,
    max_prompt_length=256,
    use_mps_device=(device == "mps"),
    fp16=False,
    bf16=False,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_config,
    train_dataset=dataset,
    tokenizer=dpo_tokeniser,
    peft_config=dpo_lora_config,
)

print("DPO trainer configured.")
print(f"beta = {dpo_config.beta}  (KL divergence penalty - controls how far the model drifts)")

In [ ]:
print("Running DPO training...")
dpo_trainer.train()

# Save the DPO-aligned model
dpo_trainer.save_model(str(MODEL_DIR / "tinyllama-dpo" / "final"))
print("DPO training complete. Aligned model saved.")

## 5. Seeing the difference

Finally we ask the aligned model a question and look at how it answers. The point of the whole notebook is that this answer should feel more specific, better structured, and more confident than what the plain base model would have given.


In [ ]:
def generate_response(model, tokeniser, prompt, max_new_tokens=200):
    messages = [
        {"role": "system", "content": "You are a financial analyst assistant."},
        {"role": "user", "content": prompt}
    ]
    formatted = tokeniser.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokeniser(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.3,
            do_sample=True, pad_token_id=tokeniser.eos_token_id
        )
    return tokeniser.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


test_prompt = "What are the most important things to look at when analysing a company's annual report?"

print(f"Prompt: {test_prompt}")
print("=" * 70)

print("\n[Aligned model response]:")
print("-" * 40)
print(generate_response(dpo_trainer.model, dpo_tokeniser, test_prompt))

## Recap

In this notebook we:

- Saw what alignment is for: getting answers people actually prefer, not just correct ones
- Built a preference dataset of chosen and rejected answer pairs
- Trained a small reward model, just to understand the idea behind the older approach
- Ran DPO, the simpler modern method, to align the model
- Met the `beta` setting, which balances following the preferences against staying close to the original model

We now have a model that has been adapted to the domain by fine-tuning, and then aligned to better answers by DPO. The one thing left is to actually measure whether all of this helped, which is the final notebook.

Next up: notebook 06, Evaluation.
